# Atividade de RAG para Livros

O objetivo dessa atividade é utilizar os conhecimentos sobre RAG aprendidos em sala para criar um pipeline de RAG completo (exceto pela parte de Agentes).

O livro escolhido foi "Meditações" de Marco Aurélio. Isso porque, além de ser uma obra muito interessante por conter os pensamentos íntimos e filosóficos de um dos maiores imperadores de Roma, também possui uma estrutura previsível: Marco Aurélio enumera seus pensamentos em formato de lições e os separa em 12 livros.

*Atividade feita pelo aluno Luiz Gustavo*

# Instalações
Caso seja preciso, instale as bibliotecas nesse ambiente.

In [1]:
# Instalando bibliotecas
# %pip install -q \
#     python-dotenv \
#     requests \
#     gradio \
#     transformers \
#     pymupdf \
#     faiss-cpu \
#     langchain-core \
#     langchain-community \
#     langchain-experimental \
#     langchain-huggingface \
#     langchain-openai \
#     langchain-text-splitters

# Bibliotecas

In [2]:
# Importando as bibliotecas

# 1. Bibliotecas Padrão do Python
import logging
import os
import re
import time
import uuid
import warnings

# 2. Bibliotecas de Terceiros (Gerais)
from dotenv import find_dotenv, load_dotenv
import gradio as gr
import requests
import transformers

# 3. Ecossistema LangChain
# Langchain Core
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory

# Langchain Community
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS

# Langchain - Integrações e Ferramentas Específicas
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Variáveis de Ambiente

As variáveis de ambiente estão salvas na mesma pasta que esse notebook. Caso seja necessário, atualize a chave da API da OpenAI. Ela foi usada apenas na LLM final que gera os resultados durante a conversa.

In [3]:
print('Procurando as variáveis de ambiente...')

# Procurando as variáveis de ambiente na mesma pasta que o notebook
if load_dotenv(find_dotenv()):
    print('Arquivo de variáveis de ambiente encontrado!')
else:
    raise FileNotFoundError('Arquivo de variáveis de ambiente não encontrado!\nVerifique se o .env se encontra na mesma pasta do notebook.')    

Procurando as variáveis de ambiente...
Arquivo de variáveis de ambiente encontrado!


# Download do Livro
Nessa atividade vamos utilizar apenas um livro, então poderiamos ter apenas colocado ele na mesma pasta do notebook, mas para ser mais engenhoso usamos o requests para download do livro, por ser uma biblioteca muito confiável para requisições HTTP.

In [4]:
# URL e nome que será gravado no arquivo
url = "https://masculinistaopressoroficial.wordpress.com/wp-content/uploads/2017/06/meditac3a7c3b5es-marco-aurc3a9lio.pdf"
file_name = "meditacoes_marco_aurelio.pdf"

# Verifica se o arquivo já existe
if os.path.exists(file_name):
    file_path = os.path.abspath(file_name)
    print(f"O arquivo '{file_name}' já existe em: {file_path}")
    print("Pulando o download...")
else: # Se não existir, faz o download
    print(f"Iniciando o download do livro {file_name} em: {url}")
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status() 

        with open(file_name, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
                
        file_path = os.path.abspath(file_name)
        print(f"Download concluído com sucesso! Arquivo salvo como: {file_path}")

    except Exception as e: # Caso ocorra algum erro
        print(f"Ocorreu um erro ao baixar o arquivo: {e}")

O arquivo 'meditacoes_marco_aurelio.pdf' já existe em: C:\Users\luizg\Documents\anaconda_projects\RAG para livros\meditacoes_marco_aurelio.pdf
Pulando o download...


# Loader
Para carregar os documentos usamos o PyMuPDFLoader pela sua rapidez, isso se deve em parte pela sua implementação em C e seu gerenciamento eficiente de memória.

In [5]:
# Carregando o texto do pdf
loader = PyMuPDFLoader(file_path)
content = loader.load()

# Exemplo de metadados
print("Metadados:")
for key, value in content[0].metadata.items():
    print(f"{key}: {value}")

Metadados:
producer: SAMBox 1.0.34.RELEASE (www.sejda.org)
creator: PDFsam Basic v3.2.5.RELEASE
creationdate: 
source: C:\Users\luizg\Documents\anaconda_projects\RAG para livros\meditacoes_marco_aurelio.pdf
file_path: C:\Users\luizg\Documents\anaconda_projects\RAG para livros\meditacoes_marco_aurelio.pdf
total_pages: 85
format: PDF 1.5
title: 
author: 
subject: 
keywords: 
moddate: 2017-02-07T12:18:20+00:00
trapped: 
modDate: D:20170207121820+00'00'
creationDate: 
page: 0


## Exemplo do Conteúdo
Vamos checar o resultado contido no nosso documento...

In [6]:
# Exemplo do texto
text_example = content[5].page_content
print(f"Exemplo de texto cru:\n\n{text_example!r}")

Exemplo de texto cru:

'Livro I\n1. Aprendi com meu avô Verus: o bom caráter e a serenidade.\n2. Da reputação e memória legadas por meu pai: o caráter discreto e viril.\n3. De minha mãe: o respeito aos deuses, a generosidade e a abstenção não somente do agir mal, como\ntambém de incorrer em semelhante pensamento; mais ainda, a frugalidade no regime de vida e o\ndistanciamento do modo de viver próprio dos ricos.\n4. Do meu bisavô: o não haver frequentado as escolas públicas e ter desfrutado de bons mestres em\ncasa, e ter compreendido que, para tais fins, é preciso gastar com generosidade.\n5. Do meu preceptor: o não ter pertencido à facção nem dos Verdes, nem dos Azuis, nem partidário\ndos Grandes-Escudos, nem dos Pequenos-Escudos; o suportar as fatigas e ter poucas necessidades; o\ntrabalho com esforço pessoal e a abstenção de excessivas tarefas, e a desfavorável acolhida à\ncalúnia.\n6. De Diogneto: o evitar inúteis ocupações; e a desconfiança do que contam os que fazem prodígios e\n

Podemos ver que o texto está com uma formatação que não nos favorece, apresentando quebras de linha em lugares que não deveriam existir, isso provavelmente se deve pelo fato dessas quebras ocorrerem no final das linhas do pdf. 

# Limpeza do texto
Um parte importante do RAG é garantir que o texto está formatado de forma correta. Se a gente passar o texto do jeito que ele está para o modelo que cria os chunks, eles terão uma formatação confusa com quebras de linha no meio de frases.

Vamos criar uma função que utiliza regex para limpar o texto.

In [7]:
def clean_text(text: str) -> str: # Se a linha for curta e não tiver pontuação, assumimos que é um título e mantemos o \n.
    def substitution_rule(match):
        line = match.group(1)
        # Se a linha termina em pontuação ou é curta mantemos o \n
        if re.search(r'[.!?]$', line) or len(line) < 25:
            return line + '\n'
        # Caso contrário, transforma o \n em espaço (meio de parágrafo)
        return line + ' '
        
    # O regex r'^(.*)\n' pega cada linha individualmente
    # re.MULTILINE faz o '^' reconhecer o início de cada linha, não só do texto todo
    result = re.sub(r'^(.+)\n', substitution_rule, text, flags=re.MULTILINE)
    
    return result

In [8]:
# Testando
print(f"Exemplo de texto limpo (como o modelo vai ler):\n\n{clean_text(text_example)!r}")
print(f"\nOu então...\n\nExemplo de texto limpo (como nós lemos):\n\n{clean_text(text_example)}")

Exemplo de texto limpo (como o modelo vai ler):

'Livro I\n1. Aprendi com meu avô Verus: o bom caráter e a serenidade.\n2. Da reputação e memória legadas por meu pai: o caráter discreto e viril.\n3. De minha mãe: o respeito aos deuses, a generosidade e a abstenção não somente do agir mal, como também de incorrer em semelhante pensamento; mais ainda, a frugalidade no regime de vida e o distanciamento do modo de viver próprio dos ricos.\n4. Do meu bisavô: o não haver frequentado as escolas públicas e ter desfrutado de bons mestres em casa, e ter compreendido que, para tais fins, é preciso gastar com generosidade.\n5. Do meu preceptor: o não ter pertencido à facção nem dos Verdes, nem dos Azuis, nem partidário dos Grandes-Escudos, nem dos Pequenos-Escudos; o suportar as fatigas e ter poucas necessidades; o trabalho com esforço pessoal e a abstenção de excessivas tarefas, e a desfavorável acolhida à calúnia.\n6. De Diogneto: o evitar inúteis ocupações; e a desconfiança do que contam os que

Nota-se que o texto não termina quando deveria terminar, isso também se deve ao fato de que ele se encontra entre páginas do pdf, ou seja, o resto da lição 8 está na próxima página do pdf. 

Para resolver isso podemos unificar todo o texto em um documento só. Vamos também remover a primeira e útlima página do pdf, já que elas contém informações sobre a publicação desse livro específico. Isso não é relevante para essa atividade, queremos criar um especialista na obra em si e não nessa versão publicada do livro.

In [9]:
# Juntando o conteúdo de todas as páginas em uma string só
complete_text = "\n".join([doc.page_content for doc in content[2:-2]])

# Agora sim, a função de limpeza no texto_completo
final_text = clean_text(complete_text)

print(f"{final_text[4937:8050]}...")

Livro I
1. Aprendi com meu avô Verus: o bom caráter e a serenidade.
2. Da reputação e memória legadas por meu pai: o caráter discreto e viril.
3. De minha mãe: o respeito aos deuses, a generosidade e a abstenção não somente do agir mal, como também de incorrer em semelhante pensamento; mais ainda, a frugalidade no regime de vida e o distanciamento do modo de viver próprio dos ricos.
4. Do meu bisavô: o não haver frequentado as escolas públicas e ter desfrutado de bons mestres em casa, e ter compreendido que, para tais fins, é preciso gastar com generosidade.
5. Do meu preceptor: o não ter pertencido à facção nem dos Verdes, nem dos Azuis, nem partidário dos Grandes-Escudos, nem dos Pequenos-Escudos; o suportar as fatigas e ter poucas necessidades; o trabalho com esforço pessoal e a abstenção de excessivas tarefas, e a desfavorável acolhida à calúnia.
6. De Diogneto: o evitar inúteis ocupações; e a desconfiança do que contam os que fazem prodígios e feiticeiros sobre encantamentos e inv

Podemos ver que agora o texto é contínuo. Com essa abordagem perdemos os metadados que seriam passados para a resposta da LLM. Sendo assim, convém criarmos nossos próprios documentos com base nos tópicos do livro (Livros I até XII).

*Poderiamos também adicionar diversos outros metadados, mas para essa atividade vamos nos ater ao básico nesse quesito.*

In [10]:
def docs_splitter(text):
    # Procura por "Introdução" ou "Livro [Núm Romano]" no início da linha
    pattern = r'\n(?=(Introdução|Livro\s[IVX]+)\n)'
    splits = re.split(pattern, text)
    docs_splitted = []
    
    i = 1
    while i < len(splits):
        title = splits[i].strip()
        content = splits[i+1].strip()
        
        # Limpa o começo do texto ("Livro I\n"...)
        if content.startswith(title):
            content = content[len(title):].strip()
            
        # Pula a introdução
        if title.lower() == "introdução":
            i += 2
            continue 
            
        # Padrão para separar números seguidos de ponto: "1. ", "2. ", etc.
        pattern_licao = r'(?:^|\n)\s*(\d+)\.\s+'
        splits_licoes = re.split(pattern_licao, content)
        
        # Como não temos prefácio, começamos direto nas lições
        j = 1
        while j < len(splits_licoes):
            num_licao = splits_licoes[j]
            text_licao = splits_licoes[j+1].strip()
            
            doc = Document(
                page_content=f"{num_licao}. {text_licao}",
                metadata={
                    "origem": "meditacoes_marco_aurelio.pdf",
                    "secao": title,
                    "licao": num_licao, 
                    "autor": "Marco Aurélio",
                    "estilo": "Estoicismo"
                }
            )
            docs_splitted.append(doc)
            j += 2
                
        i += 2
        
    return docs_splitted

In [11]:
# Usando a função
book_docs = docs_splitter(final_text)

# Testando o resultado
print(f"Total de splits identificados: {len(book_docs)}")
print(f"Metadados da primeira seção: {book_docs[0].metadata}")

Total de splits identificados: 481
Metadados da primeira seção: {'origem': 'meditacoes_marco_aurelio.pdf', 'secao': 'Livro I', 'licao': '1', 'autor': 'Marco Aurélio', 'estilo': 'Estoicismo'}


## Exemplo Final dos Documentos

In [12]:
# Verificando as seções capturadas
print(f"Exemplo de 2 documentos:\n")

for i, doc in enumerate(book_docs[:2]):
    title = doc.metadata.get('secao')
    print(f"---DOCUMENTO {i}---")
    print(f"TÍTULO: {title}")
    print(f"METADADOS: {doc.metadata}")
    print(f"CONTEÚDO: {doc.page_content[:]}")
    print("-" * 100 + "\n")

Exemplo de 2 documentos:

---DOCUMENTO 0---
TÍTULO: Livro I
METADADOS: {'origem': 'meditacoes_marco_aurelio.pdf', 'secao': 'Livro I', 'licao': '1', 'autor': 'Marco Aurélio', 'estilo': 'Estoicismo'}
CONTEÚDO: 1. Aprendi com meu avô Verus: o bom caráter e a serenidade.
----------------------------------------------------------------------------------------------------

---DOCUMENTO 1---
TÍTULO: Livro I
METADADOS: {'origem': 'meditacoes_marco_aurelio.pdf', 'secao': 'Livro I', 'licao': '2', 'autor': 'Marco Aurélio', 'estilo': 'Estoicismo'}
CONTEÚDO: 2. Da reputação e memória legadas por meu pai: o caráter discreto e viril.
----------------------------------------------------------------------------------------------------



# Chunking
Como não sabemos qual estratégia de chunking será a melhor para esse livro, vamos testar algumas.

## Recursive Chunking
Nela vamos utilizar Regex e caracteres específicos para delimitar onde os chuncks devem começar.

In [13]:
print("Método 1: Recursive Chunking")

# Instancia do recursive splitter
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=[
        r"\n(?=\d+\.\s)", 
        "\n\n",           
        "\n",             
        ". ",             
        " ",              
        ""                
    ],
    is_separator_regex=True
)

chunks_recursive = recursive_splitter.split_documents(book_docs)

print(f"Concluído! Total de chunks (Recursive): {len(chunks_recursive)}")

# Para inspecionar um exemplo:
print(f"\nExemplo: {chunks_recursive[0].page_content}")

Método 1: Recursive Chunking
Concluído! Total de chunks (Recursive): 557

Exemplo: 1. Aprendi com meu avô Verus: o bom caráter e a serenidade.


Podemos ver que até funciona, mas como temos um tamanho fixo de chunk, podem existir trechos que não separam bem cada lição de Marco Aurélio. Isso seria em parte contornado pelo chunk_overlap, mas suspeito que trechos muito grandes poderiam ficar muito picotados e não serem recuperados pelo modelo final.

## Semantic Chunking
Pode ser uma boa alternativa por consegiur capturar o contexto das lições e até mesmo separar elas caso sejam grandes demais e com assuntos diferentes.
Ao invés de usar o modelo da API da OpenAI, decidi usar um modelo rodando localmente no meu PC. Esse modelo é baseado no BERT e foi treinado especificamente para textos que são parecidos em contexto.

*o normalize_encodings foi utilizado para que os resultados sejam normalizados, isso vai ajudar no futuro, já que vamos usar o FAISS, que utiliza da distância euclidiana para dar o score dos chuncks.*

In [14]:
print("Método 2: Semantic Chunkig\n")
print("Carregando o modelo de embeddings local")

# Silencia apenas avisos de carregamento de pesos do Transformers
transformers.utils.logging.set_verbosity_error()

# Inicializa o modelo de embeddings local
embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    encode_kwargs={'normalize_embeddings': True}
)
print("Modelo carregado! Configurando o divisor semântico...")

# Configura o Semantic Chunker com o modelo local
semantic_splitter = SemanticChunker(
    embeddings=embeddings_model,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=50 
)

print("Dividindo os documentos semanticamente e preservando metadados...")
# Utilizamos split_documents para manter os metadados intactos
chunks_semantic = semantic_splitter.split_documents(book_docs)

print("\nChunking concluído!")
print(f"Total de chunks gerados: {len(chunks_semantic)}\n")

print("Inspecionando os 10 primeiros chunks:\n")
print("=" * 60)

# Pegamos apenas os 10 primeiros
for i, chunk in enumerate(chunks_semantic[:10]):
    # Extrai o texto e os metadados do objeto Document
    chunk_text = chunk.page_content
    metadata = chunk.metadata
    print(f"CHUNK {i + 1} | Tamanho: {len(chunk_text)} caracteres")
    print(f"METADADOS: {metadata}")
    print("-" * 60)
    print(chunk_text)
    print("\n" + "=" * 60 + "\n")

Método 2: Semantic Chunkig

Carregando o modelo de embeddings local


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo carregado! Configurando o divisor semântico...
Dividindo os documentos semanticamente e preservando metadados...

Chunking concluído!
Total de chunks gerados: 1291

Inspecionando os 10 primeiros chunks:

CHUNK 1 | Tamanho: 59 caracteres
METADADOS: {'origem': 'meditacoes_marco_aurelio.pdf', 'secao': 'Livro I', 'licao': '1', 'autor': 'Marco Aurélio', 'estilo': 'Estoicismo'}
------------------------------------------------------------
1. Aprendi com meu avô Verus: o bom caráter e a serenidade.


CHUNK 2 | Tamanho: 74 caracteres
METADADOS: {'origem': 'meditacoes_marco_aurelio.pdf', 'secao': 'Livro I', 'licao': '2', 'autor': 'Marco Aurélio', 'estilo': 'Estoicismo'}
------------------------------------------------------------
2. Da reputação e memória legadas por meu pai: o caráter discreto e viril.


CHUNK 3 | Tamanho: 242 caracteres
METADADOS: {'origem': 'meditacoes_marco_aurelio.pdf', 'secao': 'Livro I', 'licao': '3', 'autor': 'Marco Aurélio', 'estilo': 'Estoicismo'}
--------------

Já nesse caso, vemos que ele conseguiu captar o contexto do texto e separou trechos grandes (as lições grandes foram cortadas por contexto mas nesse exemplo não é mostrado)

# Comparação das estratégias de chunking
Vamos ver quais foram os resultados dos chunks...

*Função retirada dos notebooks da aula*

In [15]:
def analisar_chunks(chunks: list, nome: str = "Análise") -> None:
    tamanhos = [len(c) if isinstance(c, str) else len(c.page_content) for c in chunks]

    print(f"\n📊 {nome}:")
    print(f"   Total de chunks: {len(tamanhos)}")
    print(f"   Menor chunk:     {min(tamanhos)} chars")
    print(f"   Maior chunk:     {max(tamanhos)} chars")
    print(f"   Média:           {sum(tamanhos)//len(tamanhos)} chars")

    # Distribuição simples
    buckets = {"<100": 0, "100-300": 0, "300-500": 0, "500-1000": 0, ">1000": 0}
    for t in tamanhos:
        if t < 100: buckets["<100"] += 1
        elif t < 300: buckets["100-300"] += 1
        elif t < 500: buckets["300-500"] += 1
        elif t < 1000: buckets["500-1000"] += 1
        else: buckets[">1000"] += 1

    print(f"\n   Distribuição:")
    for bucket, count in buckets.items():
        bar = "█" * count
        print(f"   {bucket:>10}: {bar} ({count})")

analisar_chunks(chunks_recursive, "Recursive Chunking")
analisar_chunks(chunks_semantic, "Semantic Chunkig")


📊 Recursive Chunking:
   Total de chunks: 557
   Menor chunk:     27 chars
   Maior chunk:     1000 chars
   Média:           418 chars

   Distribuição:
         <100: ██████████████████████████████████████████████ (46)
      100-300: ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████ (210)
      300-500: ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████ (124)
     500-1000: ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████ (170)
        >1000: ███████ (7)

📊 Semantic Chunkig:
   Total de chunks: 1291
   Menor chunk:     2 chars
   Maior chunk:     1568 chars
   Média:           171 chars

   Distribuição:
         <100: █████████████

O método semântico obteve mais chunks, sendo que a maioria está até 300 chars, vamos usar ele.

# Embeddings + Vector Database
Para essa atividade foram considerados o ChromaDB e o FAISS, o motivo principal de escolha pelo FAISS foi desejo pessoal do autor do código de testar esse DB. Em outro projeto pessoal o autor já utilizou o ChromaDB, além disso como o FAISS foi criado com C++, é esperado que ele tenha uma performance rápida.

Os embeddings são feitos pelo modelo que já foi instânciado, utilizando os chuncks híbridos e salvos no FAISS.

In [16]:
# Define o nome da pasta onde o banco ficará guardado
folder_path = "faiss_db"

# Define o caminho do arquivo principal do FAISS para garantir que ele foi salvo
index_file = os.path.join(folder_path, "index.faiss")

print("Verificando o banco de dados vetorial...")

# Verifica se a pasta existe e o arquivo do banco está lá dentro
if os.path.exists(folder_path) and os.path.exists(index_file):
    
    print(f"Banco de dados encontrado na pasta '{folder_path}'!")
    print("Carregando os vetores do disco direto para a memória...")
    
    # Carrega o banco existente
    vector_store = FAISS.load_local(
        folder_path, 
        embeddings_model, 
        allow_dangerous_deserialization=True
    )
    print("Banco carregado com sucesso!")

else:
    print("Nenhum banco local encontrado. Iniciando o processamento inédito...")
    print("Lendo os chunks e gerando os embeddings matemáticos (isso pode levar um minuto)...")
    
    # Cria o banco a partir do zero usando os chunks e o modelo
    vector_store = FAISS.from_documents(chunks_semantic, embeddings_model, normalize_L2=True)
    
    print("Processamento concluído! Salvando os dados localmente para o futuro...")
    
    # O método save_local cria a pasta automaticamente se ela não existir
    vector_store.save_local(folder_path)
    
    print(f"Banco de dados criado, indexado e salvo na pasta '{folder_path}'!")

# Confirmação final do tamanho do banco que está na memória pronto para uso
print("-" * 60)
print(f"Status atual: {vector_store.index.ntotal} vetores prontos para busca.")

Verificando o banco de dados vetorial...
Banco de dados encontrado na pasta 'faiss_db'!
Carregando os vetores do disco direto para a memória...
Banco carregado com sucesso!
------------------------------------------------------------
Status atual: 1291 vetores prontos para busca.


## Teste de Retrieval
Vamos ver o estado atual do RAG, como já temos os documentos, chunks, embeddings e vector database, podemos testar se ao fazer perguntas básicas, o pipeline consegue recuperar a informação contida na database.

Para avaliação da qualidade do retrieval foi usada a busca por similaridade com score. No caso do FAISS, ela usa da distância euclidiana para calcular o score, então quanto mais próximo de 0, melhor é o resultado.

In [17]:
# A pergunta que o usuário faria
query = "O que ele diz sobre a morte?"

print(f"Buscando no banco de vetores por: '{query}'\n")

start = time.time()

# O método similarity_search transforma a pergunta em vetor e retorna os 'k' resultados mais próximos
answer = vector_store.similarity_search_with_score(query, k=3)

end = time.time()

time_ms = (end - start) * 1000 # Converte de segundos para milissegundos

print(f"Tempo de resposta do FAISS: **{time_ms:.2f} milissegundos**!\n")

for i, (doc, score) in enumerate(answer):
    print(f"TOP {i+1} RESULTADO | Score (Distância L2): **{score:.4f}**")
    print(f"METADADOS: {doc.metadata}")
    print("-" * 125)
    print(doc.page_content)
    print("\n" + "=" * 125 + "\n")

Buscando no banco de vetores por: 'O que ele diz sobre a morte?'

Tempo de resposta do FAISS: **22.00 milissegundos**!

TOP 1 RESULTADO | Score (Distância L2): **0.6967**
METADADOS: {'origem': 'meditacoes_marco_aurelio.pdf', 'secao': 'Livro VI', 'licao': '28', 'autor': 'Marco Aurélio', 'estilo': 'Estoicismo'}
-----------------------------------------------------------------------------------------------------------------------------
28. A morte é o descanso da reação sensitiva, do impulso instintivo que nos move como fantoches, da evolução do pensamento, do tributo que nos impõe a carne.


TOP 2 RESULTADO | Score (Distância L2): **0.7518**
METADADOS: {'origem': 'meditacoes_marco_aurelio.pdf', 'secao': 'Livro VII', 'licao': '35', 'autor': 'Marco Aurélio', 'estilo': 'Estoicismo'}
-----------------------------------------------------------------------------------------------------------------------------
Então, tampouco tal homem considerará terrível a morte? De forma alguma.


TOP 3 RESU

# Retireval + Augmentation + Generation
Agora podemos implementar o pipeline completo. Vamos testar alguns tipos de RAG e tentar melhorar com cada iteração do pipeline.

## RAG Básico

In [18]:
# Definindo a LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3) # Temperatura 0.3 para respostas precisas mas sem um tom "seco"

# Retriever básico
basic_retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# Prompt simples
basic_prompt = ChatPromptTemplate.from_messages([
    ("system", """Você é um especialista em Marco Aurélio. 
    Responda com base no contexto abaixo:
    
    Contexto:
    {context}"""),
    ("human", "{question}")
])

# Pipeline LCEL básico (Sem memória)
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

basic_rag_pipeline = (
    {"context": basic_retriever | format_docs, "question": RunnablePassthrough()}
    | basic_prompt
    | llm
    | StrOutputParser()
)

print("Pipeline Básico (Naive RAG) carregado!")

Pipeline Básico (Naive RAG) carregado!


In [19]:
# Teste do pipeline básico
print("INICIANDO TESTE DO PIPELINE BÁSICO\n")
print("="*60)
print("CAPACIDADE ESPERADA: Responder perguntas diretas com base no contexto recuperado do vector database.")
print("-" * 60)

# Teste 1: Pergunta direta e completa
query = "O que Marco Aurélio diz sobre a morte?"
print(f"\n👤 USUÁRIO: {query}")

# O pipeline básico busca os 3 documentos mais próximos e o LLM responde.
response = basic_rag_pipeline.invoke(query)

print(f"\n🤖 BOT (Naive RAG): {response}")
print("="*60)

INICIANDO TESTE DO PIPELINE BÁSICO

CAPACIDADE ESPERADA: Responder perguntas diretas com base no contexto recuperado do vector database.
------------------------------------------------------------

👤 USUÁRIO: O que Marco Aurélio diz sobre a morte?

🤖 BOT (Naive RAG): Marco Aurélio, em suas reflexões, aborda a morte de maneira filosófica e serena, enfatizando a inevitabilidade e a naturalidade desse fenômeno. Ele sugere que a morte deve ser encarada como um descanso da reação sensitiva e dos impulsos instintivos que nos controlam. Para ele, a morte não é um fim trágico, mas uma transformação, parte do ciclo natural da vida.

Em seus escritos, ele também menciona a importância de lembrar-se daqueles que viveram intensamente e que, apesar de suas mortes, deixaram um legado. Isso serve como um lembrete de que a vida é efêmera e que devemos valorizar o tempo que temos. A citação de Heráclito sobre a transformação dos elementos naturais ilustra a ideia de que a morte é apenas uma mudança de

In [20]:
# Teste 2: Pergunta sem relação com o livro
print("="*60)
query = "Como trocar o pneu do carro?"
print(f"\n👤 USUÁRIO: {query}")

# O pipeline básico busca os 3 documentos mais próximos e o LLM responde.
response = basic_rag_pipeline.invoke(query)

print(f"\n🤖 BOT (Naive RAG): {response}")
print("="*60)


👤 USUÁRIO: Como trocar o pneu do carro?

🤖 BOT (Naive RAG): Trocar o pneu do carro é uma habilidade útil e pode ser feita seguindo alguns passos simples. Aqui está um guia passo a passo:

1. **Prepare-se**:
   - Estacione o carro em uma superfície plana e segura.
   - Acione o freio de estacionamento.
   - Coloque um triângulo de sinalização se estiver em uma estrada.

2. **Reúna os materiais**:
   - Esteja com o estepe, o macaco, a chave de roda e, se possível, luvas.

3. **Remova a roda danificada**:
   - Se o carro tiver calotas, remova-as com a chave de roda.
   - Afrouxe os parafusos da roda (gire no sentido anti-horário) antes de levantar o carro, mas não os retire completamente.
   - Coloque o macaco sob o ponto de apoio do carro (consulte o manual do veículo para a localização correta).
   - Levante o carro até que a roda esteja suspensa.

4. **Troque o pneu**:
   - Remova completamente os parafusos da roda danificada e guarde-os em um lugar seguro.
   - Retire a roda danifica

In [21]:
# Teste 3: Memória
print("="*60)
query = "Qual foi a primeira pergunta que te fiz?"
print(f"\n👤 USUÁRIO: {query}")

# O pipeline básico busca os 3 documentos mais próximos e o LLM responde.
response = basic_rag_pipeline.invoke(query)

print(f"\n🤖 BOT (Naive RAG): {response}")
print("="*60)


👤 USUÁRIO: Qual foi a primeira pergunta que te fiz?

🤖 BOT (Naive RAG): A primeira pergunta que você me fez foi: "Qual foi a primeira pergunta que te fiz?"


Nota-se que o pipeline simples consegue recuperar a informação, melhorar e passar ela para a LLM, que consegue dar uma resposta coesa. Porém ele não traz citações de onde essa informação veio. Ao ser perguntada sobre algo que não está na database, ela simplesmente traz a informação, mesmo não sendo relevante para o contexto da tarefa. Quando perguntado sobre qual foi a primeira pergunta feita, ele também não sabe qual foi e apenas inventa a pergunta, isso porque ele não tem memória.

Vamos melhorar o pipeline, adicionando um threshold para perguntas que tem baixa relevância e citações.

## RAG Melhorado (Threshold + Citações)

In [22]:
# Silencia o warning de relevância matemática do LangChain
warnings.filterwarnings("ignore", message="Relevance scores must be between 0 and 1")

# Retriever blindado com threshold
shielded_retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 3,
        "score_threshold": 0.3
    }
)

# Formatador de citações numéricas
def format_numbered_docs(docs):
    """Transforma os chunks do FAISS em um texto com marcadores [1], [2], etc."""
    context = ""
    for i, doc in enumerate(docs, 1):
        book_section = doc.metadata.get('secao', 'Desconhecido')
        context += f"[{i}] ({book_section}): {doc.page_content}\n\n"
    return context

# Prompt especialista rigoroso
scholar_prompt = ChatPromptTemplate.from_messages([
    ("system", """Você é um acadêmico especialista nas Meditações de Marco Aurélio.
    Sua ÚNICA fonte de verdade são os [Documentos Numerados] fornecidos abaixo.
    Se a resposta não estiver nos documentos, diga explicitamente: 'Marco Aurélio não aborda isso nos trechos recuperados'.
    Ao fazer uma afirmação, você DEVE incluir a citação correspondente no final da frase, no formato [N].
    
    [Documentos Numerados]:
    {context}"""),
    ("human", "{question}")
])

# Pipeline intermediário para reter a resposta e os documentos brutos
shielded_rag_pipeline = (
    RunnablePassthrough.assign(raw_docs=lambda x: shielded_retriever.invoke(x["question"]))
    | RunnablePassthrough.assign(context=lambda x: format_numbered_docs(x["raw_docs"]))
    | RunnablePassthrough.assign(answer=(scholar_prompt | llm | StrOutputParser()))
)

# Função auxiliar para anexar as referências no final da resposta do bot
def ask_scholar(query):
    result = shielded_rag_pipeline.invoke({"question": query})
    
    final_response = result["answer"]
    retrieved_docs = result["raw_docs"]
    
    # Se encontrou documentos, formata as referências
    if retrieved_docs:
        final_response += "\n\n" + "-"*60 + "\nREFERÊNCIAS CONSULTADAS:\n"
        for i, doc in enumerate(retrieved_docs, 1):
            book_section = doc.metadata.get('secao', 'Desconhecido')
            lesson_number = doc.metadata.get('licao', 'Desconhecido')
            snippet = doc.page_content[:].replace("\n", " ")
            final_response += f"  [{i}] {book_section} | Lição {lesson_number} - \"{snippet}\"\n"
            
    return final_response

print("Pipeline com Threshold, Citações e Referências carregado!")

Pipeline com Threshold, Citações e Referências carregado!


In [23]:
# Teste do pipeline melhorado
print("INICIANDO TESTE DO PIPELINE MELHORADO\n")
print("="*60)
print("CAPACIDADE ESPERADA: Responder perguntas diretas com base no contexto recuperado do vector database, adicionando citações e não respondendo perguntas não relacionadas ao assunto do livro.")
print("-" * 60)

# Teste 1: Pergunta direta e completa
query = "O que Marco Aurélio diz sobre a morte?"
print(f"\n👤 USUÁRIO: {query}")

# O pipeline básico busca os 3 documentos mais próximos e o LLM responde.
response = ask_scholar(query)

print(f"\n🤖 BOT (RAG com citações e threshold): {response}")
print("="*60)

INICIANDO TESTE DO PIPELINE MELHORADO

CAPACIDADE ESPERADA: Responder perguntas diretas com base no contexto recuperado do vector database, adicionando citações e não respondendo perguntas não relacionadas ao assunto do livro.
------------------------------------------------------------

👤 USUÁRIO: O que Marco Aurélio diz sobre a morte?

🤖 BOT (RAG com citações e threshold): Marco Aurélio aborda a morte de várias maneiras em suas Meditações. Ele sugere que um remédio eficaz para menosprezar a morte é lembrar-se das pessoas que se apegaram à vida, refletindo que todos, eventualmente, enfrentam a morte, independentemente de suas experiências de vida [1]. Além disso, ele descreve a morte como um estado de descanso da reação sensitiva e dos impulsos instintivos que nos movem, indicando que é uma parte natural da existência [2]. Ele também menciona a visão de Heráclito sobre a morte como uma transformação, onde cada forma de vida se converte em outra, ressaltando a continuidade e a mudança 

In [24]:
# Teste 2: Pergunta sem relação com o livro
print("="*60)
query = "Como trocar o pneu do carro?"
print(f"\n👤 USUÁRIO: {query}")

# O pipeline básico busca os 3 documentos mais próximos e o LLM responde.
response = ask_scholar(query)

print(f"\n🤖 BOT (RAG com citações e threshold): {response}")
print("="*60)

No relevant docs were retrieved using the relevance score threshold 0.3



👤 USUÁRIO: Como trocar o pneu do carro?

🤖 BOT (RAG com citações e threshold): Marco Aurélio não aborda isso nos trechos recuperados.


In [25]:
# Teste 3: Memória
print("="*60)
query = "Qual foi a primeira pergunta que te fiz?"
print(f"\n👤 USUÁRIO: {query}")

# O pipeline básico busca os 3 documentos mais próximos e o LLM responde.
response = ask_scholar(query)

print(f"\n🤖 BOT (RAG com citações e threshold): {response}")
print("="*60)


👤 USUÁRIO: Qual foi a primeira pergunta que te fiz?

🤖 BOT (RAG com citações e threshold): Marco Aurélio não aborda isso nos trechos recuperados.

------------------------------------------------------------
REFERÊNCIAS CONSULTADAS:
  [1] Livro X | Lição 37 - "37. Em toda ação feita por qualquer um, acostume- te, na medida de tuas possibilidades, a perguntar- te: “Com que finalidade realiza essa ação?”. Mas começa primeiro contigo mesmo."



Agora melhorou, mas ainda não temos memória.

## RAG Final (Memória + Reformulação + Threshold + Citações)

In [26]:
# Revisor de perguntas (Query Reformulation)
reformulation_prompt = ChatPromptTemplate.from_messages([
    ("system", """Dada a conversa abaixo e uma nova pergunta, reformule a pergunta para que ela seja 
    independente e completa por si só, contendo todo o contexto necessário para uma busca. 
    Se a pergunta já for independente, retorne-a exatamente como está. 
    NÃO responda à pergunta, apenas reformule-a."""),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

reformulation_chain = reformulation_prompt | llm | StrOutputParser()

def get_standalone_question(inputs):
    # Só reformula se houver histórico de conversa
    if inputs.get("history"):
        new_question = reformulation_chain.invoke(inputs)
        print(f"[Bastidores] Pergunta reformulada: '{new_question}'")
        return new_question
    return inputs["question"]

# Prompt com memória
scholar_prompt_with_history = ChatPromptTemplate.from_messages([
    ("system", """Você é um acadêmico especialista nas Meditações de Marco Aurélio.
    Sua ÚNICA fonte de verdade são os [Documentos Numerados] fornecidos abaixo.
    NUNCA invente uma citação. Se a resposta não estiver nos documentos, diga explicitamente: 'Marco Aurélio não aborda isso nos trechos recuperados'.
    Ao fazer uma afirmação, você DEVE incluir a citação correspondente no final da frase, no formato [N].
    
    [Documentos Numerados]:
    {context}"""),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

# Pipeline LCEL final
advanced_rag_pipeline = (
    RunnablePassthrough.assign(standalone_question=get_standalone_question)
    | RunnablePassthrough.assign(raw_docs=lambda x: shielded_retriever.invoke(x["standalone_question"]))
    | RunnablePassthrough.assign(context=lambda x: format_numbered_docs(x["raw_docs"]))
    | RunnablePassthrough.assign(answer=(scholar_prompt_with_history | llm | StrOutputParser()))
)

# Gerenciador de memória do LangChain
session_store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]

# Empacota o pipeline com o injetor/extrator de memória
emperor_assistant = RunnableWithMessageHistory(
    advanced_rag_pipeline,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
    output_messages_key="answer" # Garante que apenas a resposta final vá para o histórico
)

# Função de interação
def ask_emperor_with_memory(session_id, query):
    """Executa o pipeline completo e formata as referências no final."""
    session_config = {"configurable": {"session_id": session_id}}
    
    result = emperor_assistant.invoke({"question": query}, config=session_config)
    
    final_response = result["answer"]
    retrieved_docs = result["raw_docs"]
    
    # Se encontrou documentos, anexa as referências
    if retrieved_docs:
        final_response += "\n\n" + "-"*60 + "\nREFERÊNCIAS CONSULTADAS:\n"
        for i, doc in enumerate(retrieved_docs, 1):
            book_section = doc.metadata.get('secao', 'Desconhecido')
            lesson_number = doc.metadata.get('licao', 'Desconhecido')
            snippet = doc.page_content[:].replace("\n", " ")
            if lesson_number == "Desconhecido":
                final_response += f"  [{i}] {book_section} - \"{snippet}\"\n"
            else:
                final_response += f"  [{i}] {book_section} | Lição {lesson_number} - \"{snippet}\"\n"
            
    return final_response

print("RAG Final (Memória + Reformulação + Threshold + Citações) carregado!")

RAG Final (Memória + Reformulação + Threshold + Citações) carregado!


In [27]:
# Teste do pipeline melhorado
print("INICIANDO TESTE DO PIPELINE FINAL\n")
print("="*60)
print("CAPACIDADE ESPERADA: Responder perguntas diretas com base no contexto recuperado do vector database, adicionando citações, não respondendo a perguntas não relacionadas ao assunto do livro, tendo memória e reformulando as perguntas.")
print("-" * 60)

# Teste 1: Pergunta direta e completa
query = "O que Marco Aurélio diz sobre lidar com a morte?"
print(f"\n👤 USUÁRIO: {query}")

# O pipeline básico busca os 3 documentos mais próximos e o LLM responde.
response = ask_emperor_with_memory("teste_jupyter", query)

print(f"\n🤖 BOT (RAG Final): {response}")
print("="*60)

INICIANDO TESTE DO PIPELINE FINAL

CAPACIDADE ESPERADA: Responder perguntas diretas com base no contexto recuperado do vector database, adicionando citações, não respondendo a perguntas não relacionadas ao assunto do livro, tendo memória e reformulando as perguntas.
------------------------------------------------------------

👤 USUÁRIO: O que Marco Aurélio diz sobre lidar com a morte?

🤖 BOT (RAG Final): Marco Aurélio aborda a morte como uma parte natural da vida e sugere que devemos lembrar daqueles que já partiram para menosprezar o medo da morte. Ele menciona que, ao refletir sobre aqueles que se apegaram à vida, percebemos que todos acabarão enfrentando a morte, independentemente de suas ações em vida [1]. Além disso, ele descreve a morte como um descanso da reação sensitiva e dos impulsos instintivos, enfatizando que é uma transição que todos devem aceitar [2]. Por fim, ele nos incentiva a refletir sobre nossas ações e a questionar se a morte é realmente terrível, já que ela não 

In [28]:
# Teste 2: Pergunta sem relação com o livro
print("="*60)
query = "Como trocar o pneu do carro?"
print(f"\n👤 USUÁRIO: {query}")

# O pipeline básico busca os 3 documentos mais próximos e o LLM responde.
response = ask_emperor_with_memory("teste_jupyter", query)

print(f"\n🤖 BOT (RAG Final): {response}")
print("="*60)


👤 USUÁRIO: Como trocar o pneu do carro?


No relevant docs were retrieved using the relevance score threshold 0.3


[Bastidores] Pergunta reformulada: 'Quais são os passos necessários para trocar o pneu de um carro, incluindo as ferramentas e precauções que devem ser tomadas durante o processo?'

🤖 BOT (RAG Final): Marco Aurélio não aborda isso nos trechos recuperados.


In [29]:
# Teste 3: Memória
print("="*60)
query = "Qual foi a primeira pergunta que te fiz?"
print(f"\n👤 USUÁRIO: {query}")

# O pipeline básico busca os 3 documentos mais próximos e o LLM responde.
response = ask_emperor_with_memory("teste_jupyter", query)

print(f"\n🤖 BOT (RAG Final): {response}")
print("="*60)


👤 USUÁRIO: Qual foi a primeira pergunta que te fiz?
[Bastidores] Pergunta reformulada: 'A primeira pergunta que você fez foi: "O que Marco Aurélio diz sobre lidar com a morte?"'

🤖 BOT (RAG Final): A primeira pergunta que você fez foi: "O que Marco Aurélio diz sobre lidar com a morte?"

------------------------------------------------------------
REFERÊNCIAS CONSULTADAS:
  [1] Livro IV | Lição 50 - "50. Remédio simples, mas eficaz, para menosprezar a morte, é lembrar-se dos que se apegaram com tenacidade à vida. O que mais têm em relação aos que morreram prematuramente? Em qualquer caso, jazem em alguma parte Ceciliano, Fábio, Juliano, Lépido e outros como eles, que a tantos levaram à tumba, para serem também eles levados depois."
  [2] Livro X | Lição 29 - "29. Detenha particularmente em cada uma das ações que praticas e te pergunte se a morte for terrível porque te priva disso."
  [3] Livro IX | Lição 39 - "Pergunta à tua alma: “Morreste? Foste destruída?"



# Conversando com o "especialista"

Existem métricas que podem ser usadas para avaliar um pipeline de RAG, elas não foram implementadas nessa atividade, mas podemos conversar com o modelo para ter uma sensação de como ele performa.

In [30]:
# =====================================================================
# WRAPPER PRINCIPAL DO CHAT
# =====================================================================

# Imagens
avatar_bot  = "https://images.unsplash.com/photo-1739323147107-bc748671f498?q=80&w=987&auto=format&fit=crop"
avatar_user = "https://images.unsplash.com/photo-1680995823172-ac8cd6db0de5?q=80&w=1070&auto=format&fit=crop"

def chat_interface_wrapper(user_message, gradio_history, session_state):
    if not gradio_history:
        session_state["id"] = str(uuid.uuid4())

    session_config = {"configurable": {"session_id": session_state["id"]}}

    full_text = ""
    documents = []

    for chunk in emperor_assistant.stream({"question": user_message}, config=session_config):
        if "raw_docs" in chunk:
            documents = chunk["raw_docs"]
        if "answer" in chunk:
            full_text += chunk["answer"]
            yield full_text

    cited_in_order = list(dict.fromkeys(
        int(n) for n in re.findall(r'\[(\d+)\]', full_text)
    ))

    if documents and cited_in_order:
        refs_html = (
            "\n\n<details>\n"
            "<summary>📚 <b>Referências consultadas (clique para expandir)</b></summary>\n"
            "<blockquote>\n"
        )
        for i in cited_in_order:
            doc           = documents[i - 1]
            section       = doc.metadata.get("secao", "Desconhecida")
            lesson        = doc.metadata.get("licao")
            clean_content = doc.page_content.replace("\n", " ")
            header        = f"{section} | Lição {lesson}" if lesson else section
            refs_html    += f"<b>[{i}] {header}</b><br><i>\"{clean_content}\"</i><br><br>"

        refs_html += "</blockquote>\n</details>"
        full_text += refs_html
        yield full_text

# =====================================================================
# INTERFACE
# =====================================================================

print("Iniciando o Oráculo...")

demo = gr.ChatInterface(
    fn=chat_interface_wrapper,
    title="🏛️ Oráculo de Marco Aurélio",
    description="<div style='text-align: center'>Converse com um especialista nas <i>Meditações</i>. O sistema utiliza RAG com memória isolada por sessão e citações das lições originais.</div>",
    chatbot=gr.Chatbot(
        avatar_images=[avatar_user, avatar_bot],
        height=520,
        render_markdown=True,
    ),
    additional_inputs=[gr.State({"id": str(uuid.uuid4())})],
    additional_inputs_accordion=gr.Accordion(visible=False),
    examples=[
        ["O que Marco Aurélio diz sobre a morte?"],
        ["Como ele diz sobre lidar com pessoas difíceis?"],
        ["O que ele pensa sobre a passagem do tempo?"],
        ["Qual a primeira pergunta que eu fiz?"],
    ],
)

demo.launch(share=True)

Iniciando o Oráculo...
* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


# Conclusão

Ao longo desta atividade, construímos um pipeline completo baseado na arquitetura **RAG (Retrieval-Augmented Generation)**. Este método permitiu que a inteligência artificial consultasse uma base de conhecimento externa e específica antes de gerar suas respostas, em vez de depender apenas do seu treinamento geral. O processo envolveu desde o carregamento e limpeza complexa do PDF de *Meditações*, passando por diferentes estratégias de *chunking* e geração de *embeddings* locais, até o armazenamento no banco vetorial e a criação de uma interface interativa com memória.

### Arquitetura e Tecnologias Utilizadas
Para dar vida ao "Assistente do Imperador", as seguintes ferramentas e modelos foram orquestrados:

* **Arquitetura:** RAG (Retrieval-Augmented Generation) com pipeline gerenciado pelo ecossistema **LangChain**.
* **Modelo de Embeddings:** Utilizamos o modelo local da Hugging Face, **`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`**. Ele atuou como nosso "bibliotecário", responsável por transformar tanto os trechos do livro (gerados pelo *Semantic Chunker*) quanto as perguntas do usuário em vetores matemáticos para a busca por similaridade.
* **Banco de Dados Vetorial (DB Final):** O **FAISS (Facebook AI Similarity Search)** foi o banco de dados escolhido para armazenar esses vetores e recuperar rapidamente os fragmentos de texto mais relevantes para cada pergunta.
* **Modelo de Linguagem (LLM):** O "escritor" do nosso sistema foi o **`gpt-4o-mini`** da OpenAI. Ele recebeu os trechos de texto recuperados pelo FAISS e os utilizou para redigir respostas fluídas e contextualizadas.
* **Interface:** A interação com o usuário foi construída utilizando o **Gradio**, permitindo uma experiência de chat fluída com histórico de conversas e exibição das referências consultadas.

### Avaliação Prática e Desafios do RAG
Apesar de o sistema estar totalmente funcional, a interação prática com a obra de Marco Aurélio revelou lições valiosas sobre o comportamento imprevisível das LLMs e os desafios inerentes ao RAG:

* **Acertos e Precisão:** Em grande parte das interações, o sistema brilha. Ele consegue recuperar as lições corretamente, conectá-las e gerar respostas ricas e embasadas. Também acerta em cheio ao barrar temas que realmente não existem na obra, ativando corretamente a trava do limite de relevância (*threshold*).
* **Alucinação vs. Prompt:** Mesmo com um *system prompt* extremamente restritivo ordenando que o modelo use **apenas** os documentos fornecidos, a LLM por vezes desobedece. Em certas ocasiões, ela ignora a falta de contexto recuperado e usa seu conhecimento prévio (treinamento original do GPT) para inventar respostas plausíveis sobre o Estoicismo, fugindo da regra de gerar citações.
* **Falsos Negativos (Falha de Recuperação):** Ocorreram situações em que o modelo afirmou categoricamente que *Marco Aurélio não aborda isso nos trechos recuperados*, mesmo sabendo que a resposta existe no livro. Isso evidencia que a busca por similaridade semântica ou o nosso limite de corte (*score threshold*) falhou em capturar os *chunks* corretos para aquela pergunta específica.
* **Inconsistências na Memória:** A janela de histórico se mostrou uma faca de dois gumes. Enquanto funciona muito bem para dar continuidade a um raciocínio, ela pode causar confusão em interações iniciais. Se a primeira pergunta do usuário exigir memória (ex: *"Qual foi a minha primeira pergunta?"*), o modelo tem comportamentos mistos: às vezes ele compreende que a conversa acabou de começar, mas em outras ele alucina e inventa um histórico prévio que nunca ocorreu.

### Bateria de Testes Estratégicos

Para avaliar e validar todas essas nuances do pipeline (busca semântica, síntese, memória e filtros de segurança), estabelecemos cinco cenários de testes fundamentais:

**1. A pergunta clássica de aplicação prática (Testa a busca semântica básica)**
> **Pergunta:** *"Como Marco Aurélio aconselha que devemos lidar com pessoas difíceis, ingratas ou invejosas logo pela manhã?"*

> **O que foi esperado:** O bot deve trazer a famosa reflexão (geralmente do Livro II) onde o imperador diz para lembrarmos que essas pessoas agem assim por ignorância sobre o que é o bem e o mal.

> **O que ele respondeu:** "Marco Aurélio aconselha que, ao lidarmos com pessoas difíceis, ingratas ou invejosas, devemos nos preparar mentalmente para essas interações logo pela manhã. Ele sugere que devemos lembrar que essas características são parte da natureza humana e que não devemos nos deixar afetar por elas. Em suas meditações, ele enfatiza a importância de manter a calma e a razão, reconhecendo que não podemos controlar as ações dos outros, mas podemos controlar nossa resposta a elas. Essa preparação mental nos ajuda a enfrentar o dia com uma atitude mais equilibrada e resiliente [N]."

Conclusão: **Inventou a resposta**.

**2. A pergunta sobre um conceito-chave (Testa a síntese de múltiplos trechos)**
> **Pergunta:** *"O que significa, na prática, 'viver de acordo com a natureza'?"*

> **O que esperar:** Como esse é um tema recorrente, o bot deve puxar diferentes partes do livro (várias referências [1], [2]) e sintetizar que não se trata de ir morar no mato, mas sim de usar a razão (que é a natureza humana) e aceitar o destino.

> **O que ele respondeu:** "Viver de acordo com a natureza, segundo Marco Aurélio, significa agir em conformidade com a nossa própria natureza racional e sociável. Isso envolve seguir o bom caminho, não consentir com o falso ou incerto, e vincular nossos instintos a ações que sejam úteis à comunidade. Além disso, implica em desejar e detestar apenas aquilo que depende exclusivamente de nós e aceitar tudo o que a natureza comum nos designa. Assim, viver de acordo com a natureza é alinhar nossas ações e desejos com a razão e a justiça, reconhecendo nossa parte na natureza maior" [2].
>
> 📚 Referências consultadas (clique para expandir)
[2] Livro VIII | Lição 7
"Toda natureza está satisfeita consigo mesma quando segue o bom caminho. E segue o bom caminho a natureza racional quando em suas imaginações não dá seu consentimento nem ao falso nem ao incerto e, em troca, vincula seus instintos somente a ações úteis à comunidade, quando se dedica a desejar e detestar aquelas coisas que dependem exclusivamente de nós, e abraça tudo o que lhe designa a natureza comum. Pois é uma parte dela, da mesma forma que a natureza da folha é parte da natureza da planta, com exceção de que, nesse caso, a natureza da folha é parte de uma natureza insensível, desprovida de razão e capaz de ser interrompida, enquanto que a natureza do homem é parte de uma natureza livre de obstáculos, inteligente e justa, se é que naturalmente distribui a todos com equidade e segundo o mérito, sua parte de tempo, substância, causa, energia, acidente."
 
Conclusão: **Resposta coerente**.

**3. A pergunta de continuidade (Testa a Memória e a Reformulação)**
> **Pergunta:** *"E como eu posso aplicar esse mesmo pensamento quando estiver me sentindo exausto ou com dor?"*

> **O que esperar:** O bot precisa olhar para a pergunta anterior (viver segundo a natureza ou lidar com pessoas), entender o contexto, reformular a pergunta nos bastidores e buscar o que o livro diz sobre suportar a dor física e o cansaço.

**Aqui ele exibe um comportamento interessante:** Se antes dessa pergunta, forem feitas as perguntas 1 e 2, ele da uma resposta que até poderia ser interpretada como correta, mas que em si não é uma referência direta a dor. Se essa for a primeira pergunta, ele da uma resposta que é diretamente relacionada a dor. Isso indica que ele usa também o histórico da conversa para responder algumas perguntas.

Respostas:

> **O que ele respondeu após a pergunta 1 e 2:** "Quando você se sentir exausto ou com dor, Marco Aurélio sugere que você deve observar atentamente o que sua natureza exige nesse momento. A aceitação do que a natureza pede é fundamental, desde que isso não comprometa sua índole de ser racional. Isso significa que, mesmo em momentos de dificuldade, você deve buscar compreender e aceitar sua condição, sem se deixar dominar pelo sofrimento. Em vez de resistir ou se revoltar contra a dor, é importante reconhecer que ela faz parte da experiência humana e que sua reação a ela deve ser guiada pela razão. Assim, você pode encontrar um equilíbrio e manter sua paz interior, mesmo diante de desafios físicos [2]."
>
>📚 Referências consultadas (clique para expandir)
[2] Livro X | Lição 2
"2. Observa atentamente o que exige tua natureza, na convicção de que só ela te governa; continuando, ponha em prática e aceita isso que te exige desde que não sofra a tua índole de ser vivo. Seguidamente, deves observar o que pede a tua natureza de ser vivo, dessa forma, deves aceitar o que exige, a não ser que corrompa a tua natureza de ser racional, que é também um ser sociável."

> **O que ele responde se for a primeira pergunta:** "Marco Aurélio sugere que, quando você se sentir aborrecido ou em dor, deve lembrar-se de que cedeu à dor. Essa reflexão pode ajudar a reorientar sua mente e a lidar com a exaustão ou a dor de maneira mais consciente. Ao reconhecer que a dor é uma parte da experiência humana e que você tem o poder de escolher como reagir a ela, você pode encontrar um certo alívio e clareza em momentos difíceis [1]."
>
>📚 Referências consultadas (clique para expandir)
[1] Livro VII | Lição 64
"Depois, sempre que te aborreças com alguma dessas coisas, diga para ti mesmo: cedi à dor."

Conclusão: **O grau de coesão depende da interpretação do leitor e do momento que a pergunta é feita**.

**4. A pergunta sobre o próprio autor (Testa os metadados e trechos biográficos)**
> **Pergunta:** *"O que ele diz ter aprendido com sua família, especificamente com seu pai e sua mãe?"*

> **O que esperar:** O bot deve focar quase inteiramente no Livro I, onde Marco Aurélio lista as qualidades que herdou e aprendeu com seus mentores e parentes.

> **O que ele respondeu:** "Marco Aurélio menciona que aprendeu com seu pai a mansidão e a firmeza serena nas decisões que foram profundamente examinadas. Ele também destaca a importância de não se vangloriar com as honras aparentes, o amor ao trabalho e à perseverança, além da disposição para escutar aqueles que podem contribuir de forma útil para a comunidade. Com sua mãe, ele aprendeu a devoção e a bondade, o que também moldou seu caráter e suas ações [1]."
>
>📚 Referências consultadas (clique para expandir)
[1] Livro I | Lição 16
"16. De meu pai: a mansidão e a firmeza serena nas decisões profundamente examinadas. O não vangloriar-se com as honras aparentes; o amor ao trabalho e à perseverança; o estar disposto a escutar aos que podiam contribuir de forma útil para a comunidade."

Conclusão: **Resposta coesa**.

**5. A pegadinha "Fora de Contexto" (Testa o Threshold/Filtro de Relevância)**
> **Pergunta:** *"Quais são os conselhos do imperador sobre como investir dinheiro para ficar rico rapidamente?"*

> **O que esperar:** O estoicismo despreza a busca cega por riquezas, e o livro foca na alma e na moral. Mais importante ainda: os termos financeiros modernos não estão no texto. Seu sistema de segurança (Threshold de 0.4) deve barrar a busca e o bot deve responder com firmeza: "Marco Aurélio não aborda isso nos trechos recuperados."

> **O que ele respondeu:** "Marco Aurélio não aborda isso nos trechos recuperados."

Conclusão: **Resposta coesa**.

### Próximos Passos
Esta atividade demonstra que criar um RAG vai muito além de apenas "conectar peças". O RAG é um processo iterativo. Para melhorar os resultados observados, trabalhos futuros poderiam envolver o ajuste refinado do *chunk size*, a troca do modelo de *embeddings* por um mais poderoso em língua portuguesa, ou a implementação de uma arquitetura de Agentes (onde o modelo tem o poder de avaliar a própria resposta e buscar novamente no banco caso perceba que alucinou).

# Avaliação com RAGAS

In [31]:
# Conjunto de 10 perguntas e respostas de referência (ground_truth)
# Baseadas integralmente no livro "Meditações" de Marco Aurélio.

perguntas_teste = [
    "O que Marco Aurélio afirma ter aprendido com sua mãe?",
    "Qual é o conselho de Marco Aurélio ao despertar a aurora sobre os encontros do dia a dia?",
    "De acordo com o Livro II, quais são os três elementos que compõem o ser humano?",
    "Qual é o refúgio mais tranquilo e com mais calma que o homem pode buscar, segundo o Livro IV?",
    "O que o texto afirma sobre a glória póstuma e a fama?",
    # "Como a morte é descrita no Livro VI?",
    # "O que significa 'viver de acordo com a natureza' no Livro VIII?",
    # "Qual a definição prática de maldade dada no Livro VII?",
    # "Segundo o Livro IX, por que aquele que persegue os prazeres e evita as fadigas comete uma impiedade?",
    # "Qual foi a atitude de Epicuro durante a enfermidade, citada por Marco Aurélio como exemplo?"
]

ground_truths = [
    "Ele aprendeu o respeito aos deuses, a generosidade e a abstenção não somente do agir mal, como também de incorrer em semelhante pensamento; além da frugalidade no regime de vida e o distanciamento do modo de viver próprio dos ricos.",
    "Ele aconselha a fazer a consideração prévia de que encontrará pessoas indiscretas, ingratas, insolentes, mentirosas, invejosas e não-sociáveis, e que isso lhes ocorre por ignorância sobre o que é o bem e o mal.",
    "O homem é constituído apenas por um pouco de carne, um breve fôlego vital e o guia interior.",
    "Em nenhuma parte o homem se retira com maior tranquilidade do que em sua própria alma, sendo este o refúgio que se encontra no pequeno campo de si mesmo.",
    "A fama póstuma é considerada esquecimento e vaidade total. O homem que se deslumbra com ela não imagina que todos os que o lembrarem também morrerão em breve, sendo a terra inteira apenas um ponto.",
    # "A morte é o descanso da reação sensitiva, do impulso instintivo que move as pessoas como fantoches, da evolução do pensamento e do tributo que a carne impõe.",
    # "Significa agir em conformidade com a natureza racional, não dando consentimento ao falso nem ao incerto, vinculando os instintos a ações úteis à comunidade, desejando o que depende de nós e abraçando o que a natureza comum designa.",
    # "A maldade é aquilo que já se viu muitas vezes. De baixo para cima, são as mesmas coisas habituais e efêmeras que enchem as histórias antigas e contemporâneas.",
    # "Porque o homem inevitavelmente recriminará a natureza comum por distribuir males e bens de forma injusta entre bons e maus, o que é agir em discordância com a indiferença da natureza universal.",
    # "Epicuro, quando enfermo, não falava de seus sofrimentos corporais nem dava aos médicos a oportunidade de se exibir, mas continuava se ocupando dos princípios naturais e de como a inteligência segue imperturbável atendendo a seu próprio bem."
]

print(f"📋 Dataset de avaliação: {len(perguntas_teste)} perguntas")
print()
for i, (p, gt) in enumerate(zip(perguntas_teste, ground_truths), 1):
    print(f"{i}. Pergunta: {p}")
    print(f"   Ground Truth: {gt[:80]}...")
    print()

📋 Dataset de avaliação: 5 perguntas

1. Pergunta: O que Marco Aurélio afirma ter aprendido com sua mãe?
   Ground Truth: Ele aprendeu o respeito aos deuses, a generosidade e a abstenção não somente do ...

2. Pergunta: Qual é o conselho de Marco Aurélio ao despertar a aurora sobre os encontros do dia a dia?
   Ground Truth: Ele aconselha a fazer a consideração prévia de que encontrará pessoas indiscreta...

3. Pergunta: De acordo com o Livro II, quais são os três elementos que compõem o ser humano?
   Ground Truth: O homem é constituído apenas por um pouco de carne, um breve fôlego vital e o gu...

4. Pergunta: Qual é o refúgio mais tranquilo e com mais calma que o homem pode buscar, segundo o Livro IV?
   Ground Truth: Em nenhuma parte o homem se retira com maior tranquilidade do que em sua própria...

5. Pergunta: O que o texto afirma sobre a glória póstuma e a fama?
   Ground Truth: A fama póstuma é considerada esquecimento e vaidade total. O homem que se deslum...



In [32]:
%pip install -q datasets

Note: you may need to restart the kernel to use updated packages.


In [33]:
import pandas as pd
from datasets import Dataset

# Estruturando no formato Dataset do HuggingFace (padrão do RAGAS)
eval_data = {
    "question": perguntas_teste,
    "ground_truth": ground_truths
}

eval_dataset = Dataset.from_dict(eval_data)
eval_df = eval_dataset.to_pandas()

# Exibindo as primeiras linhas para validação no notebook
display(eval_df.head())

,question,ground_truth
0,O que Marco Aurélio afirma ter aprendido com s...,"Ele aprendeu o respeito aos deuses, a generosi..."
1,Qual é o conselho de Marco Aurélio ao desperta...,Ele aconselha a fazer a consideração prévia de...
2,"De acordo com o Livro II, quais são os três el...",O homem é constituído apenas por um pouco de c...
3,Qual é o refúgio mais tranquilo e com mais cal...,Em nenhuma parte o homem se retira com maior t...
4,O que o texto afirma sobre a glória póstuma e ...,A fama póstuma é considerada esquecimento e va...


In [34]:
# Executando o RAG para cada pergunta e coletando respostas + contextos
print("⏳ Executando o sistema RAG em todas as perguntas...")
print()

respostas_geradas = []
contextos_recuperados = []

for i, pergunta in enumerate(perguntas_teste, 1):
    # Cria uma sessão limpa para cada pergunta da avaliação
    session_id = f"eval_session_{uuid.uuid4()}"
    session_config = {"configurable": {"session_id": session_id}}
    
    # O pipeline emperor_assistant recebe um dict com 'question'
    resultado = emperor_assistant.invoke({"question": pergunta}, config=session_config)

    # Coletando a resposta final (no seu código está na chave "answer")
    respostas_geradas.append(resultado["answer"])
    
    # Coletando o conteúdo dos documentos (no seu código está em "raw_docs")
    docs_recuperados = [doc.page_content for doc in resultado["raw_docs"]]
    contextos_recuperados.append(docs_recuperados)

    print(f"✅ [{i}/{len(perguntas_teste)}] {pergunta[:]}...")

print(f"\n✅ {len(respostas_geradas)} respostas geradas!")

# O zip une as listas índice por índice
for i, (pergunta, gerada, esperada, contextos) in enumerate(zip(perguntas_teste, respostas_geradas, ground_truths, contextos_recuperados), 1):
    print(f"\n--- Pergunta {i} ---")
    print(f"❓ Pergunta: {pergunta}")
    print(f"🤖 Resposta gerada: {gerada}")
    print(f"🎯 Resposta esperada: {esperada}")
    print(f"📚 Contextos recuperados: {len(contextos)} documentos")

⏳ Executando o sistema RAG em todas as perguntas...

✅ [1/5] O que Marco Aurélio afirma ter aprendido com sua mãe?...
✅ [2/5] Qual é o conselho de Marco Aurélio ao despertar a aurora sobre os encontros do dia a dia?...
✅ [3/5] De acordo com o Livro II, quais são os três elementos que compõem o ser humano?...
✅ [4/5] Qual é o refúgio mais tranquilo e com mais calma que o homem pode buscar, segundo o Livro IV?...
✅ [5/5] O que o texto afirma sobre a glória póstuma e a fama?...

✅ 5 respostas geradas!

--- Pergunta 1 ---
❓ Pergunta: O que Marco Aurélio afirma ter aprendido com sua mãe?
🤖 Resposta gerada: Marco Aurélio não aborda isso nos trechos recuperados.
🎯 Resposta esperada: Ele aprendeu o respeito aos deuses, a generosidade e a abstenção não somente do agir mal, como também de incorrer em semelhante pensamento; além da frugalidade no regime de vida e o distanciamento do modo de viver próprio dos ricos.
📚 Contextos recuperados: 1 documentos

--- Pergunta 2 ---
❓ Pergunta: Qual é o con

In [35]:
%pip install -q ragas nest_asyncio

Note: you may need to restart the kernel to use updated packages.


In [36]:
import os
import asyncio
import contextvars
import nest_asyncio
import pandas as pd
from openai import OpenAI
from ragas import EvaluationDataset, SingleTurnSample, evaluate
from ragas.llms import llm_factory
from ragas.metrics._faithfulness import Faithfulness
from ragas.metrics._answer_relevance import AnswerRelevancy
from ragas.metrics._context_recall import ContextRecall
from ragas.metrics._context_precision import LLMContextPrecisionWithReference
from ragas.metrics._context_entities_recall import ContextEntityRecall
from ragas.metrics._answer_similarity import AnswerSimilarity
from ragas.metrics._factual_correctness import FactualCorrectness
from ragas.metrics._noise_sensitivity import NoiseSensitivity
from ragas import RunConfig

nest_asyncio.apply()

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
lc_embeddings = embeddings_model

# ==============================================================
# O NOVO WRAPPER BLINDADO
# ==============================================================
class AsyncEmbeddingsWrapper:
    def __init__(self, lc_embeddings):
        self._lc = lc_embeddings
    
    async def aembed_text(self, text: str):
        # A MÁGICA AQUI: Pega o loop atual, cria um contexto limpo e 
        # joga o processamento do embedding para uma thread isolada.
        # Isso impede 100% o erro "cannot enter context".
        loop = asyncio.get_event_loop()
        ctx = contextvars.Context()
        return await loop.run_in_executor(None, ctx.run, self._lc.embed_query, text)

ragas_embeddings = AsyncEmbeddingsWrapper(lc_embeddings)
ragas_llm = llm_factory("gpt-4o-mini", client=openai_client)

samples = [
    SingleTurnSample(
        user_input=q,
        response=a,
        retrieved_contexts=c,
        reference=gt,
    )
    for q, a, c, gt in zip(perguntas_teste, respostas_geradas, contextos_recuperados, ground_truths)
]

dataset_ragas = EvaluationDataset(samples=samples)

print("📊 Dataset RAGAS criado:")
print(f"   Amostras: {len(dataset_ragas)}")
print()
print("Primeira amostra:")
print(f"  user_input: {samples[0].user_input}")
print(f"  response: {samples[0].response[:100]}...")
print(f"  retrieved_contexts: {len(samples[0].retrieved_contexts)} documentos")
print(f"  reference: {samples[0].reference[:100]}...")

📊 Dataset RAGAS criado:
   Amostras: 5

Primeira amostra:
  user_input: O que Marco Aurélio afirma ter aprendido com sua mãe?
  response: Marco Aurélio não aborda isso nos trechos recuperados....
  retrieved_contexts: 1 documentos
  reference: Ele aprendeu o respeito aos deuses, a generosidade e a abstenção não somente do agir mal, como també...


In [37]:
metricas = [
    Faithfulness(llm=ragas_llm),
    AnswerRelevancy(llm=ragas_llm, embeddings=lc_embeddings),
    ContextRecall(llm=ragas_llm),
    LLMContextPrecisionWithReference(llm=ragas_llm),
    ContextEntityRecall(llm=ragas_llm),
    AnswerSimilarity(embeddings=ragas_embeddings),
    FactualCorrectness(llm=ragas_llm),
    NoiseSensitivity(llm=ragas_llm),
]

print("\n🧪 Iniciando avaliação RAGAS...")
print("   (Isso pode levar alguns minutos — RAGAS usa o LLM para cada métrica)\n")

# CONFIGURAÇÃO DE SEGURANÇA PARA EVITAR TIMEOUTS
run_config = RunConfig(timeout=600, max_workers=1, max_retries=2)

resultado_avaliacao = evaluate(
    dataset=dataset_ragas,
    metrics=metricas,
    run_config=run_config,
    raise_exceptions=False,
)

print("\n✅ Avaliação concluída!\n")


🧪 Iniciando avaliação RAGAS...
   (Isso pode levar alguns minutos — RAGAS usa o LLM para cada métrica)



Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\ProgramData\anaconda3\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000001556F8E8800> is already entered
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Task was destroyed but it is pending!
task: <Task pending name='Task-4' coro=<Kernel.shell_main() running at C:\ProgramData\anaconda3\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.__wakeup()]>
C:\ProgramData\anaconda3\Lib\http\cookiejar.py:1228: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  def deepvalues(mapping):
Task was destroyed but it is pending!
task: <Task pending name='Task-2' coro=<_async_in_context.<locals>.run_in_context() running at C:\ProgramData\anaconda3\Lib\site-packages\ipykernel\


✅ Avaliação concluída!



In [ ]:
df_resultados = resultado_avaliacao.to_pandas()

print("📊 RESULTADOS DETALHADOS POR PERGUNTA:")
print("=" * 70)

colunas_metricas = [
    "faithfulness",
    "answer_relevancy",
    "context_recall",
    "llm_context_precision_with_reference",
    "context_entity_recall",
    "answer_similarity",
    "factual_correctness",
    "noise_sensitivity",
]
colunas_disponiveis = [c for c in colunas_metricas if c in df_resultados.columns]

for i, row in df_resultados.iterrows():
    print(f"\n[{i+1}] {row['user_input']}")
    for col in colunas_disponiveis:
        valor = row.get(col, "N/A")
        if isinstance(valor, float):
            emoji = "✅" if valor >= 0.7 else ("⚠️" if valor >= 0.4 else "❌")
            print(f"     {emoji} {col}: {valor:.4f}")
        else:
            print(f"     ❓ {col}: {valor}")

In [ ]:
print("\n📈 MÉDIAS GERAIS DO SISTEMA:")
print("=" * 50)

for col in colunas_disponiveis:
    media = df_resultados[col].mean()
    emoji = "✅" if media >= 0.7 else ("⚠️" if media >= 0.4 else "❌")
    barra = "█" * int(media * 20) + "░" * (20 - int(media * 20))
    print(f"  {emoji} {col:<42} {barra} {media:.4f}")

print("\nLegenda: ✅ >= 0.7 (bom) | ⚠️ 0.4-0.7 (médio) | ❌ < 0.4 (ruim)")

In [ ]:
%pip install -q matplotlib

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df_viz = df_resultados[colunas_disponiveis].copy()
labels_curtos = {
    "faithfulness": "Faithfulness",
    "answer_relevancy": "Ans. Relevancy",
    "context_recall": "Ctx. Recall",
    "llm_context_precision_with_reference": "Ctx. Precision",
    "context_entity_recall": "Entity Recall",
    "answer_similarity": "Ans. Similarity",
    "factual_correctness": "Factual Corr.",
    "noise_sensitivity": "Noise Sensit.",
}
df_plot = df_viz.rename(columns={c: labels_curtos.get(c, c) for c in colunas_disponiveis})
perguntas_curtas = [f"Q{i+1}" for i in range(len(df_plot))]
df_plot.index = perguntas_curtas

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle("Avaliação RAGAS v3 (8 métricas)", fontsize=15, fontweight="bold", y=1.02)

# 1. Heatmap
ax1 = axes[0]
data = df_plot.values.astype(float)
im = ax1.imshow(data, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
ax1.set_xticks(range(len(df_plot.columns)))
ax1.set_xticklabels(df_plot.columns, rotation=45, ha="right", fontsize=8)
ax1.set_yticks(range(len(df_plot)))
ax1.set_yticklabels(perguntas_curtas)
ax1.set_title("Heatmap por Pergunta", fontweight="bold")
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        v = data[i, j]
        txt = f"{v:.2f}" if not np.isnan(v) else "N/A"
        color = "black" if 0.35 < v < 0.75 else "white"
        ax1.text(j, i, txt, ha="center", va="center", fontsize=7, color=color)
plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)

# 2. Bar chart das médias
ax2 = axes[1]
medias = df_plot.mean()
colors = ["#2ecc71" if v >= 0.7 else "#f39c12" if v >= 0.4 else "#e74c3c" for v in medias]
bars = ax2.bar(medias.index, medias.values, color=colors, edgecolor="white", linewidth=0.8)
ax2.axhline(0.7, color="#27ae60", linestyle="--", linewidth=1.2, label="Bom (≥0.7)")
ax2.axhline(0.4, color="#e67e22", linestyle="--", linewidth=1.2, label="Médio (≥0.4)")
ax2.set_ylim(0, 1.15)
ax2.set_title("Médias por Métrica", fontweight="bold")
ax2.set_xticklabels(medias.index, rotation=45, ha="right", fontsize=8)
ax2.legend(fontsize=8)
for bar, val in zip(bars, medias.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{val:.2f}", ha="center", va="bottom", fontsize=8, fontweight="bold")

# 3. Radar chart
ax3 = axes[2]
ax3.remove()
ax3 = fig.add_subplot(1, 3, 3, polar=True)
cats = list(medias.index)
N = len(cats)
vals = list(medias.values) + [medias.values[0]]
angles = [n / float(N) * 2 * np.pi for n in range(N)] + [0]
ax3.plot(angles, vals, "o-", linewidth=2, color="#3498db")
ax3.fill(angles, vals, alpha=0.25, color="#3498db")
ax3.set_xticks(angles[:-1])
ax3.set_xticklabels(cats, fontsize=7)
ax3.set_ylim(0, 1)
ax3.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax3.set_yticklabels(["0.2","0.4","0.6","0.8","1.0"], fontsize=6)
ax3.set_title("Radar de Desempenho", fontweight="bold", pad=15)

plt.tight_layout()
plt.savefig("ragas_v3_resultados.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico salvo em ragas_v3_resultados.png")
